In [0]:
# List all files in the restaurant_data directory
files = dbutils.fs.ls("/Volumes/restaurant_dev/london_dev_schema/raw/restaurant_data/")

# Display file information
for file in files:
    print(f"Name: {file.name}")
    print(f"Path: {file.path}")
    print(f"Size: {file.size} bytes")
    print("-" * 50)

In [0]:
# Function to get all files from a directory recursively
def get_all_files(path):
    all_files = []
    items = dbutils.fs.ls(path)
    
    for item in items:
        if item.path.endswith('/'):
            # It's a directory, recurse
            all_files.extend(get_all_files(item.path))
        else:
            # It's a file
            all_files.append(item)
    
    return all_files

# Get all files
base_path = "/Volumes/restaurant_dev/london_dev_schema/raw/restaurant_data/"
all_files = get_all_files(base_path)

print(f"Found {len(all_files)} files:")
for file in all_files:
    print(f"  - {file.name} ({file.size} bytes)")

In [0]:
# Dictionary to store all dataframes
dataframes = {}

# Read each category
for category, file_paths in sorted(file_groups.items()):
    print(f"\nLoading {category}...")
    
    if len(file_paths) == 1:
        # Single file - read directly
        df = spark.read.csv(file_paths[0], header=True, inferSchema=True)
        dataframes[category] = df
        print(f"  ✓ Loaded {df.count()} rows")
    else:
        # Multiple files - read all and union
        # Sort file paths to ensure chronological order
        sorted_paths = sorted(file_paths)
        df = spark.read.csv(sorted_paths, header=True, inferSchema=True)
        dataframes[category] = df
        print(f"  ✓ Loaded {len(file_paths)} files with {df.count()} total rows")

print(f"\n{'='*60}")
print(f"Successfully created {len(dataframes)} dataframes:")
for name in sorted(dataframes.keys()):
    print(f"  - {name}")

In [0]:
# Display schema information for each dataframe
for name, df in sorted(dataframes.items()):
    print(f"\n{'='*70}")
    print(f"Schema for: {name}")
    print(f"Row count: {df.count():,}")
    print(f"{'='*70}")
    df.printSchema()
    print(f"\nSample data (first 3 rows):")
    display(df.limit(3))

In [0]:
from pyspark.sql.functions import col, count, when, isnan, countDistinct

# Function to display comprehensive stats for a dataframe
def display_dataframe_stats(name, df):
    print(f"\n{'='*80}")
    print(f"STATISTICS FOR: {name.upper()}")
    print(f"{'='*80}")
    
    # Basic info
    print(f"\n📊 Basic Information:")
    print(f"   Total Rows: {df.count():,}")
    print(f"   Total Columns: {len(df.columns)}")
    
    # Column data types
    print(f"\n📋 Column Data Types:")
    for field in df.schema.fields:
        print(f"   {field.name:30s} -> {field.dataType}")
    
    # Null counts
    print(f"\n🔍 Null Value Analysis:")
    null_counts = df.select([
        count(when(col(c).isNull() | isnan(c), c)).alias(c) 
        if df.schema[c].dataType.simpleString() in ['double', 'float']
        else count(when(col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]).collect()[0].asDict()
    
    total_rows = df.count()
    has_nulls = False
    for column, null_count in null_counts.items():
        if null_count > 0:
            has_nulls = True
            null_pct = (null_count / total_rows) * 100
            print(f"   {column:30s} -> {null_count:,} nulls ({null_pct:.2f}%)")
    
    if not has_nulls:
        print("   ✓ No null values found in any column")
    
    # Numeric statistics
    numeric_cols = [field.name for field in df.schema.fields 
                   if field.dataType.simpleString() in ['integer', 'double', 'float', 'long', 'bigint']]
    
    if numeric_cols:
        print(f"\n📈 Numeric Column Statistics:")
        stats_df = df.select(numeric_cols).describe()
        display(stats_df)
    
    print("\n" + "-"*80)

# Display stats for all dataframes
for name in sorted(dataframes.keys()):
    display_dataframe_stats(name, dataframes[name])

In [0]:
# Assign each dataframe to a named variable
customer_reviews_df = dataframes['customer_reviews']
customers_df = dataframes['customers']
daily_operations_df = dataframes['daily_operations']
delivery_performance_df = dataframes['delivery_performance']
employees_df = dataframes['employees']
inventory_df = dataframes['inventory']
menu_items_df = dataframes['menu_items']
order_details_df = dataframes['order_details']
orders_df = dataframes['orders']
restaurants_df = dataframes['restaurants']

print("✓ All dataframes have been assigned to variables:")
print("  - customer_reviews_df")
print("  - customers_df")
print("  - daily_operations_df")
print("  - delivery_performance_df")
print("  - employees_df")
print("  - inventory_df")
print("  - menu_items_df")
print("  - order_details_df")
print("  - orders_df")
print("  - restaurants_df")

In [0]:
from collections import defaultdict
import re

# Group files by category
file_groups = defaultdict(list)

for file in all_files:
    file_name = file.name
    file_path = file.path
    
    # Determine the category
    if file_name.startswith('customer_reviews_'):
        file_groups['customer_reviews'].append(file_path)
    elif file_name == 'customers.csv':
        file_groups['customers'].append(file_path)
    elif file_name.startswith('daily_operations_'):
        file_groups['daily_operations'].append(file_path)
    elif file_name.startswith('delivery_performance_'):
        file_groups['delivery_performance'].append(file_path)
    elif file_name == 'employees.csv':
        file_groups['employees'].append(file_path)
    elif file_name.startswith('inventory_'):
        file_groups['inventory'].append(file_path)
    elif file_name == 'menu_items.csv':
        file_groups['menu_items'].append(file_path)
    elif file_name.startswith('order_details_'):
        file_groups['order_details'].append(file_path)
    elif file_name.startswith('orders_'):
        file_groups['orders'].append(file_path)
    elif file_name == 'restaurants.csv':
        file_groups['restaurants'].append(file_path)

# Display grouping summary
print("File groups summary:")
for category, files in sorted(file_groups.items()):
    print(f"  {category}: {len(files)} file(s)")